# Latent Dynamics Pipeline

Run the full workflow in one notebook:
1. Train VAE
2. Generate latent dynamics arrays
3. Characterize latent dynamics
4. Generate artificial data (uniform + qubit-wise)

Scripts are organized by purpose:
- `ML/training/` for training
- `ML/analysis_processing/` for processing, analysis, and simulation
- `ML/plotting/` for plotting and visualization


In [14]:
from pathlib import Path
import os
import subprocess
import sys

_cwd = Path.cwd().resolve()
if (_cwd / "ML").is_dir():
    ROOT = _cwd
elif _cwd.name == "ML" and (_cwd.parent / "ML").is_dir():
    ROOT = _cwd.parent
else:
    raise RuntimeError(
        f"Could not infer repo root from cwd={_cwd}. Run from repo root or ML/."
    )

ML_DIR = ROOT / "ML"
RUN_DIR = ML_DIR / "runs" / "fid_job_memory_noreset_large"
DATA_PKL = ROOT / "quantum_code" / "data" / "fid_job_memory_noreset_large_1404.pkl"

DEVICE = "auto"  # auto | cpu | cuda | mps
BATCH_SIZE = 128
EPOCHS = 100
LR = 5e-4
ANNEAL_STEPS = 25  # maps to --beta-warmup-epochs

print("CWD:", _cwd)
print("ROOT:", ROOT)
print("ML_DIR:", ML_DIR)
print("RUN_DIR:", RUN_DIR)
print("DATA_PKL:", DATA_PKL)
print("LR:", LR)
print("ANNEAL_STEPS:", ANNEAL_STEPS)


CWD: /Users/krzywdaja/Documents/quantum-gym/ML
ROOT: /Users/krzywdaja/Documents/quantum-gym
ML_DIR: /Users/krzywdaja/Documents/quantum-gym/ML
RUN_DIR: /Users/krzywdaja/Documents/quantum-gym/ML/runs/fid_job_memory_noreset_large
DATA_PKL: /Users/krzywdaja/Documents/quantum-gym/quantum_code/data/fid_job_memory_noreset_large_1404.pkl
LR: 0.0005
ANNEAL_STEPS: 25


In [12]:
def run_py(script_name: str, *args: str):
    cmd = [sys.executable, str(ML_DIR / script_name), *map(str, args)]
    env = dict(os.environ)
    root_str = str(ROOT)
    prev = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = root_str if not prev else f"{root_str}:{prev}"
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=ROOT, env=env)


## 1) Training

In [11]:
run_py(
    "training/train_vae_model.py",
    "--data", DATA_PKL,
    "--out-dir", RUN_DIR,
    "--device", DEVICE,
    "--batch-size", BATCH_SIZE,
    "--epochs", EPOCHS,
    "--lr", LR,
    "--beta-warmup-epochs", ANNEAL_STEPS,
)


$ /opt/anaconda3/bin/python /Users/krzywdaja/Documents/quantum-gym/first_tests/ML/training/train_vae_model.py --data /Users/krzywdaja/Documents/quantum-gym/first_tests/quantum_code/data/fid_job_memory_noreset_large.pkl --out-dir /Users/krzywdaja/Documents/quantum-gym/first_tests/ML/runs/fid_job_memory_noreset_large --device auto --batch-size 128 --epochs 100 --lr 0.0005 --beta-warmup-epochs 25
Device: cpu
Train: 100 epochs, batch=128, lr=0.0005, β 0.05 → 1.0 over 25 epochs


[W NNPACK.cpp:61] Could not initialize NNPACK! Reason: Unsupported hardware.


Epoch    0 | β=0.0880 | loss 28.45739 (bce 28.13097 + β·kl 0.32642)


Traceback (most recent call last):
  File "/Users/krzywdaja/Documents/quantum-gym/first_tests/ML/training/train_vae_model.py", line 17, in <module>
    main()
  File "/Users/krzywdaja/Documents/quantum-gym/first_tests/ML/train_vae.py", line 230, in main
    loss.backward()
  File "/opt/anaconda3/lib/python3.12/site-packages/torch/_tensor.py", line 525, in backward
    torch.autograd.backward(
  File "/opt/anaconda3/lib/python3.12/site-packages/torch/autograd/__init__.py", line 267, in backward
    _engine_run_backward(
  File "/opt/anaconda3/lib/python3.12/site-packages/torch/autograd/graph.py", line 744, in _engine_run_backward
    return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

## 2) Generate latent dynamics arrays

In [15]:
run_py(
    "analysis_processing/generate_latent_dynamics.py",
    "--data", DATA_PKL,
    "--source-file", DATA_PKL,
    "--ckpt", RUN_DIR / "checkpoints" / "vae_checkpoint.pt",
    "--out-dir", RUN_DIR,
    "--device", DEVICE,
)


$ /opt/anaconda3/bin/python /Users/krzywdaja/Documents/quantum-gym/ML/analysis_processing/generate_latent_dynamics.py --data /Users/krzywdaja/Documents/quantum-gym/quantum_code/data/fid_job_memory_noreset_large_1404.pkl --source-file /Users/krzywdaja/Documents/quantum-gym/quantum_code/data/fid_job_memory_noreset_large_1404.pkl --ckpt /Users/krzywdaja/Documents/quantum-gym/ML/runs/fid_job_memory_noreset_large/checkpoints/vae_checkpoint.pt --out-dir /Users/krzywdaja/Documents/quantum-gym/ML/runs/fid_job_memory_noreset_large --device auto
Wrote /Users/krzywdaja/Documents/quantum-gym/ML/runs/fid_job_memory_noreset_large/reports/latent_dynamics.json
Wrote /Users/krzywdaja/Documents/quantum-gym/ML/runs/fid_job_memory_noreset_large/data/latent_dynamics_arrays.npz
Wrote /Users/krzywdaja/Documents/quantum-gym/ML/runs/fid_job_memory_noreset_large/figures/latent_drift_covariance.png
Wrote /Users/krzywdaja/Documents/quantum-gym/ML/runs/fid_job_memory_noreset_large/figures/latent_drift_correlation.

## 3) Characterize latent dynamics

Characterization artifacts are produced by `analysis_processing/generate_latent_dynamics.py` in:
- `reports/latent_dynamics.json`
- `data/latent_dynamics_arrays.npz`
- `figures/latent_drift_*.png`

Optional: generate trajectory GIF from encoded latents.

In [16]:
run_py(
    "plotting/plot_latent_dynamics.py",
    "--run-dir", RUN_DIR,
    "--source", "encoded",
    "--source-file", DATA_PKL,
)


$ /opt/anaconda3/bin/python /Users/krzywdaja/Documents/quantum-gym/ML/plotting/plot_latent_dynamics.py --run-dir /Users/krzywdaja/Documents/quantum-gym/ML/runs/fid_job_memory_noreset_large --source encoded --source-file /Users/krzywdaja/Documents/quantum-gym/quantum_code/data/fid_job_memory_noreset_large_1404.pkl
Wrote /Users/krzywdaja/Documents/quantum-gym/ML/runs/fid_job_memory_noreset_large/figures/latent_dynamics_encoded_avg50.gif


## 4) Generate artificial data (uniform + qubit-wise)

Use `analysis_processing/generate_synthetic_zebra.py` in two latent modes:
- `iid` for uniform latent sampling across qubits
- `qubit-mean-shift` for qubit-wise shifted latents

## 5) Generate synthetic latent dynamics and plot latent_dynamics_sim_...

In [17]:
CKPT = RUN_DIR / "checkpoints" / "vae_checkpoint.pt"
DYN_JSON = RUN_DIR / "reports" / "latent_dynamics.json"

# Uniform latent sampling across qubits
#run_py(
#    "analysis_processing/generate_synthetic_zebra.#py",
#    "--ckpt", CKPT,
#    "--latent-mode", "iid",
#)

# Qubit-wise shifted latent sampling
run_py(
    "analysis_processing/generate_synthetic_zebra.py",
    "--source-file", DATA_PKL,
    "--ckpt", CKPT,
    "--latent-mode", "qubit-mean-shift",
    "--dynamics-json", DYN_JSON,
    "--qubit-latent-scale", 1.5,
)


$ /opt/anaconda3/bin/python /Users/krzywdaja/Documents/quantum-gym/ML/analysis_processing/generate_synthetic_zebra.py --source-file /Users/krzywdaja/Documents/quantum-gym/quantum_code/data/fid_job_memory_noreset_large_1404.pkl --ckpt /Users/krzywdaja/Documents/quantum-gym/ML/runs/fid_job_memory_noreset_large/checkpoints/vae_checkpoint.pt --latent-mode qubit-mean-shift --dynamics-json /Users/krzywdaja/Documents/quantum-gym/ML/runs/fid_job_memory_noreset_large/reports/latent_dynamics.json --qubit-latent-scale 1.5
Wrote /Users/krzywdaja/Documents/quantum-gym/ML/runs/fid_job_memory_noreset_large/data/vae_synthetic_fid_memory.pkl (5000 shots × 17 qubits × 50 τ)
Wrote: /Users/krzywdaja/Documents/quantum-gym/ML/runs/fid_job_memory_noreset_large/figures/vae_zebra_memory_3d_repetitions_2d_bin100.gif
Wrote: /Users/krzywdaja/Documents/quantum-gym/ML/runs/fid_job_memory_noreset_large/figures/vae_zebra_memory_3d_qubit_coclick.png
Wrote: /Users/krzywdaja/Documents/quantum-gym/ML/runs/fid_job_memory_

In [18]:
# Generate synthetic latent trajectories (sim_fitted_latent_mu_s_q.npz)
run_py(
    "analysis_processing/simulate_latent_zebra.py",
    "--ckpt", CKPT,
    "--dynamics-json", DYN_JSON,
    "--sim-mode", "ou",
    "--device", DEVICE,
)

# Plot synthetic latent dynamics GIF: latent_dynamics_sim_...
run_py(
    "plotting/plot_latent_dynamics.py",
    "--run-dir", RUN_DIR,
    "--source", "sim",
    "--device", DEVICE,
)

$ /opt/anaconda3/bin/python /Users/krzywdaja/Documents/quantum-gym/ML/analysis_processing/simulate_latent_zebra.py --ckpt /Users/krzywdaja/Documents/quantum-gym/ML/runs/fid_job_memory_noreset_large/checkpoints/vae_checkpoint.pt --dynamics-json /Users/krzywdaja/Documents/quantum-gym/ML/runs/fid_job_memory_noreset_large/reports/latent_dynamics.json --sim-mode ou --device auto
Wrote /Users/krzywdaja/Documents/quantum-gym/ML/runs/fid_job_memory_noreset_large/data/sim_fitted_fid_memory.pkl
Wrote /Users/krzywdaja/Documents/quantum-gym/ML/runs/fid_job_memory_noreset_large/reports/sim_fitted_latent_meta.json
Wrote /Users/krzywdaja/Documents/quantum-gym/ML/runs/fid_job_memory_noreset_large/data/sim_fitted_latent_mu_s_q.npz
Wrote: /Users/krzywdaja/Documents/quantum-gym/ML/runs/fid_job_memory_noreset_large/figures/sim_fitted_zebra_memory_3d_repetitions_2d_bin100.gif
Wrote: /Users/krzywdaja/Documents/quantum-gym/ML/runs/fid_job_memory_noreset_large/figures/sim_fitted_zebra_memory_3d_qubit_coclick.